# Crear y ejecutar el workflow (job) desde un notebook local
Este notebook se ejecuta en su computadora, conectado a Databricks con el perfil de la CLI. Hace cuatro cosas:
1. **Sube los notebooks** del pipeline (`src/`) a su carpeta del workspace con `databricks workspace import-dir`.
2. **Define el job** con sus tareas y dependencias: `bronze → silver → gold → feature_engineering`; luego los modelos de onda P y de amplificación en paralelo; la demanda sísmica; los modelos de daño y de perfiles en paralelo; el riesgo en Lima; y al final, en paralelo, el sismo en tiempo real y los datos del mapa de validación (la app web `mapa-validacion`, que se despliega con `Desplegar_App_Local`).
3. **Crea (o actualiza) el job** en Workflows con `databricks-sdk`.
4. **Ejecuta el job** con parámetros según el sismo que llega y muestra el resultado de cada tarea.

Como mucho corren 3 tareas a la vez.

## 1. Conexión a Databricks

In [ ]:
# Abre la sesión remota de Spark con el perfil de la CLI.
from databricks.connect import DatabricksSession

PERFIL = "22200144"
spark = DatabricksSession.builder.profile(PERFIL).serverless(True).getOrCreate()
print("Conectado a Databricks Serverless. Spark", spark.version)

## 2. Configuración

In [ ]:
# Librerías y conexión al workspace de Databricks.
import subprocess
from datetime import timedelta

import pandas as pd
from databricks.sdk import WorkspaceClient
from databricks.sdk.service import jobs

w = WorkspaceClient(profile=PERFIL)
usuario = w.current_user.me().user_name
RUTA_WORKSPACE = f"/Users/{usuario}/sismos_lakehouse"
NOMBRE_JOB = "sismos_pipeline"
CATALOGO, ESQUEMA = "grupo06proyectofinal", "sismos"
print("Usuario:", usuario, "| carpeta en el workspace:", RUTA_WORKSPACE)

## 3. Subir los notebooks al workspace
Los de `src/` (las tareas del job) y la simulación quedan en la carpeta del proyecto; los dos notebooks locales, en `locales/` como referencia.

In [ ]:
# Sube un notebook local al workspace.
def importar(archivo, destino):
    subprocess.run(["databricks", "workspace", "import", destino, "--file", archivo, "--format", "JUPYTER", "--overwrite", "-p", PERFIL], check=True)


# Sube los notebooks del proyecto al workspace.
def subir_notebooks():
    resultado = subprocess.run(["databricks", "workspace", "import-dir", "src", RUTA_WORKSPACE, "--overwrite", "-p", PERFIL],
                               capture_output=True, text=True)
    print(resultado.stdout or resultado.stderr)
    importar("Simulacion_Llegada_Datos.ipynb", f"{RUTA_WORKSPACE}/Simulacion_Llegada_Datos")
    w.workspace.mkdirs(f"{RUTA_WORKSPACE}/locales")
    for nombre in ["Subir_Datos_Databricks", "Crear_Workflow_Local", "Desplegar_App_Local"]:
        importar(f"{nombre}.ipynb", f"{RUTA_WORKSPACE}/locales/{nombre}")


subir_notebooks()

In [ ]:
# Lista los notebooks subidos.
for objeto in w.workspace.list(RUTA_WORKSPACE, recursive=True):
    print(objeto.object_type.value, objeto.path)

## 4. Tareas y dependencias del job

In [ ]:
PARAMETROS_JOB = ["fecha_desde", "fecha_hasta", "forzar_recalibrado", "recargar_maestros"]


# Define una tarea del job.
def tarea(clave, notebook, depende_de=None):
    parametros = {p: "{{job.parameters." + p + "}}" for p in PARAMETROS_JOB}
    dependencias = [jobs.TaskDependency(task_key=d) for d in (depende_de or [])]
    return jobs.Task(
        task_key=clave,
        depends_on=dependencias or None,
        notebook_task=jobs.NotebookTask(notebook_path=f"{RUTA_WORKSPACE}/{notebook}", base_parameters=parametros),
    )

In [ ]:
# Define las 12 tareas del job y sus dependencias.
tareas = [
    tarea("bronze", "01_bronze"),
    tarea("silver", "02_silver", depende_de=["bronze"]),
    tarea("gold", "03_gold", depende_de=["silver"]),
    tarea("feature_engineering", "04_feature_engineering", depende_de=["gold"]),
    tarea("ml_clasificacion", "05_ml_clasificacion", depende_de=["feature_engineering"]),
    tarea("ml_regresion", "06_ml_regresion", depende_de=["feature_engineering"]),
    tarea("demanda_sismica", "07_demanda_sismica", depende_de=["ml_regresion"]),
    tarea("ml_dano", "08_ml_dano", depende_de=["demanda_sismica"]),
    tarea("ml_clustering", "09_ml_clustering", depende_de=["demanda_sismica"]),
    tarea("riesgo_lima", "10_riesgo_lima", depende_de=["ml_dano", "ml_clustering"]),
    tarea("sismo_tiempo_real", "11_sismo_tiempo_real", depende_de=["riesgo_lima"]),
    tarea("mapa_validacion", "12_mapa_validacion", depende_de=["riesgo_lima"]),
]
print(len(tareas), "tareas:", [t.task_key for t in tareas])

## 5. Crear o actualizar el job

In [ ]:
# Crea el job o actualiza sus tareas.
def crear_o_actualizar_job():
    configuracion = dict(
        name=NOMBRE_JOB,
        max_concurrent_runs=1,
        tasks=tareas,
        parameters=[
            jobs.JobParameterDefinition(name="fecha_desde", default="2007-01-01"),
            jobs.JobParameterDefinition(name="fecha_hasta", default="2025-12-31"),
            jobs.JobParameterDefinition(name="forzar_recalibrado", default="false"),
            jobs.JobParameterDefinition(name="recargar_maestros", default="false"),
        ],
    )
    existentes = [j for j in w.jobs.list(name=NOMBRE_JOB)]
    if existentes:
        w.jobs.reset(job_id=existentes[0].job_id, new_settings=jobs.JobSettings(**configuracion))
        return existentes[0].job_id
    return w.jobs.create(**configuracion).job_id


JOB_ID = crear_o_actualizar_job()
print("Job en Workflows:", NOMBRE_JOB, "| job_id:", JOB_ID)

## 6. Funciones para ejecutar y revisar las decisiones

In [ ]:
# Lanza el job con las fechas indicadas y espera el resultado.
def ejecutar(fecha_desde, fecha_hasta, forzar_recalibrado="false", recargar_maestros="false"):
    corrida = w.jobs.run_now(job_id=JOB_ID, job_parameters={
        "fecha_desde": fecha_desde, "fecha_hasta": fecha_hasta, "forzar_recalibrado": forzar_recalibrado,
        "recargar_maestros": recargar_maestros}).result(timeout=timedelta(hours=3))
    filas = [{"tarea": t.task_key,
              "estado": t.state.result_state.value if t.state and t.state.result_state else (t.state.life_cycle_state.value if t.state else None)}
             for t in corrida.tasks]
    print(f"Corrida {corrida.run_id}: {corrida.state.result_state.value if corrida.state.result_state else corrida.state.life_cycle_state.value}")
    display(pd.DataFrame(filas))
    return corrida


# Muestra la última decisión de monitoreo de cada modelo.
def ver_decisiones():
    display(spark.table(f"{CATALOGO}.{ESQUEMA}.ml_monitoreo")
            .select("clave", "champion_version", "n_features_drift", "metrica_base", "metrica_nueva", "degradacion", "reentrenar", "motivo")
            .orderBy("clave").toPandas())
    display(spark.table(f"{CATALOGO}.{ESQUEMA}.ml_registro_modelos")
            .select("caso", "version", "algoritmo", "lote", "promovido").orderBy("caso", "version").toPandas())

## 7. Carga histórica: sismos 2007–2025
Antes se ejecuta, en Databricks, la sección 4 de `Simulacion_Llegada_Datos`.

In [ ]:
# Ejecuta la carga histórica y muestra las decisiones.
ejecutar("2007-01-01", "2025-12-31")
ver_decisiones()

## 8. Llega Ica 2026
Antes se ejecuta, en Databricks, la sección 5 de `Simulacion_Llegada_Datos`.

In [ ]:
# Ejecuta la llegada de Ica y muestra las decisiones.
ejecutar("2026-01-01", "2026-12-31")
ver_decisiones()